In [ ]:
import os
import re
import csv
from collections import defaultdict
import matplotlib.pyplot as plt
from typing import Dict, List, Union, Optional

# Helper functions


def parse_result_file(file_path: str) -> str:
    """
    Parses the result file to determine the SAT solver outcome.
    """
    try:
        with open(file_path, "r") as f:
            content = f.read().strip()
        if content == "s SATISFIABLE":
            return "SAT"
        elif content == "s UNSATISFIABLE":
            return "UNSAT"
        elif content == "c UNKNOWN":
            return "TIMEOUT"
        else:
            return "ERROR"
    except Exception as e:
        print(f"Error reading result file {file_path}: {e}")
        return "ERROR"


def parse_stderr_file(file_path: str) -> Optional[float]:
    """
    Parses the stderr file to extract execution time.
    """
    try:
        with open(file_path, "r") as f:
            content = f.read()
        # Try to parse 'real xymz' format
        match = re.search(r"real\s+(\d+)m(\d+\.\d+)s", content)
        if match:
            minutes, seconds = match.groups()
            return float(minutes) * 60 + float(seconds)
        # Alternatively, try to parse time in seconds
        match = re.search(r"Total time:\s+(\d+\.\d+)s", content)
        if match:
            return float(match.group(1))
        print(f"Could not parse time from {file_path}")
        return None
    except Exception as e:
        print(f"Error reading stderr file {file_path}: {e}")
        return None


# Result analysis function


def analyze_results(base_dir: str) -> Dict[str, Dict[str, Union[str, Optional[float]]]]:
    """
    Analyzes results in the specified directory and returns a dictionary of outcomes.
    """
    results = {}
    results_dir = os.path.join(base_dir, "results")
    stderr_dir = os.path.join(base_dir, "stderr")

    if not os.path.isdir(results_dir):
        print(f"Results directory {results_dir} does not exist.")
        return results
    if not os.path.isdir(stderr_dir):
        print(f"Stderr directory {stderr_dir} does not exist.")
        return results

    for filename in os.listdir(results_dir):
        if filename.endswith("_result.txt"):
            cnf_name = filename[: -len("_result.txt")]
            result_path = os.path.join(results_dir, filename)
            stderr_path = os.path.join(stderr_dir, f"{cnf_name}_stderr.txt")

            result = parse_result_file(result_path)
            time = parse_stderr_file(stderr_path) if os.path.exists(stderr_path) else None

            results[cnf_name] = {"result": result, "time": time}

    return results


# Result comparison function


def compare_results(
    baseline_results: Dict[str, Dict[str, Union[str, Optional[float]]]],
    other_results: Dict[str, Dict[str, Dict[str, Union[str, Optional[float]]]]],
) -> List[Dict]:
    """
    Compares baseline results with other algorithm results.
    Only samples present in the baseline are compared.
    """
    comparison = []
    for cnf_name in baseline_results.keys():
        baseline_result = baseline_results.get(cnf_name, {})
        baseline_result_value = baseline_result.get("result", "N/A")
        baseline_time_value = baseline_result.get("time", None)

        row = {
            "cnf_name": cnf_name,
            "baseline_result": baseline_result_value,
            "baseline_time": baseline_time_value,
        }

        for algo_name, results in other_results.items():
            algo_result = results.get(cnf_name, {})
            algo_result_value = algo_result.get("result", "N/A")
            algo_time_value = algo_result.get("time", None)

            row[f"{algo_name}_result"] = algo_result_value
            row[f"{algo_name}_time"] = algo_time_value

            # Validate results based on various cases
            if baseline_result_value in ["SAT", "UNSAT"]:
                if algo_result_value == baseline_result_value:
                    row[f"{algo_name}_validity"] = "VALID"
                elif algo_result_value in ["SAT", "UNSAT"]:
                    row[f"{algo_name}_validity"] = "INVALID"
                else:
                    row[f"{algo_name}_validity"] = "UNKNOWN"
            else:
                row[f"{algo_name}_validity"] = "UNKNOWN"

            # Calculate speedup
            baseline_time = baseline_time_value
            algo_time = algo_time_value
            if baseline_time is not None and baseline_time > 0 and algo_time is not None and algo_time > 0:
                if algo_result_value == "TIMEOUT":
                    row[f"{algo_name}_speedup"] = None
                else:
                    row[f"{algo_name}_speedup"] = baseline_time / algo_time
            else:
                row[f"{algo_name}_speedup"] = None

        comparison.append(row)

    return comparison


# CSV writing function


def write_comparison_csv(comparison: List[Dict], output_file: str, algo_names: List[str]):
    """
    Writes the comparison data to a CSV file.
    """
    fieldnames = ["cnf_name", "baseline_result", "baseline_time"]
    for algo in algo_names:
        fieldnames.extend([f"{algo}_result", f"{algo}_time", f"{algo}_validity", f"{algo}_speedup"])

    with open(output_file, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in comparison:
            writer.writerow(row)

    print(f"Comparison results have been written to {output_file}")


# Result plotting function


def plot_results(comparison: List[Dict], output_dir: str, algo_names: List[str]):
    """
    Generates plots for validity and speedup and saves them to the output directory.
    """
    validity_counts = {algo: defaultdict(int) for algo in algo_names}
    speedups = {algo: [] for algo in algo_names}

    for row in comparison:
        for algo in algo_names:
            validity = row.get(f"{algo}_validity", "N/A")
            validity_counts[algo][validity] += 1
            speedup = row.get(f"{algo}_speedup")
            if speedup is not None and isinstance(speedup, (int, float)):
                speedups[algo].append(speedup)

    # Speedup distribution
    for algo in algo_names:
        if speedups[algo]:
            plt.figure(figsize=(12, 6))
            plt.hist(speedups[algo], bins=20, alpha=0.7, label=algo)
            plt.xlabel("Speedup (Baseline Time / Algorithm Time)")
            plt.ylabel("Frequency")
            plt.title(f"Distribution of Speedup for {algo}")
            plt.legend()
            plt.savefig(os.path.join(output_dir, f"speedup_distribution_{algo}.png"))
            plt.close()
        else:
            print(f"No speedup data to plot for {algo}")

    # Validity pie charts
    for algo in algo_names:
        counts = validity_counts[algo]
        if counts:
            labels = list(counts.keys())
            sizes = list(counts.values())
            plt.figure(figsize=(8, 8))
            plt.pie(sizes, labels=labels, autopct="%1.1f%%")
            plt.title(f"Validity of {algo} Results")
            plt.savefig(os.path.join(output_dir, f"validity_pie_chart_{algo}.png"))
            plt.close()
        else:
            print(f"No validity data to plot for {algo}")

    # Scatter plot of execution times
    for algo in algo_names:
        times_baseline = []
        times_algo = []
        for row in comparison:
            baseline_time = row.get("baseline_time")
            algo_time = row.get(f"{algo}_time")
            algo_result = row.get(f"{algo}_result")
            if (
                baseline_time is not None
                and algo_time is not None
                and algo_time > 0
                and algo_result not in ["TIMEOUT", "N/A"]
            ):
                times_baseline.append(baseline_time)
                times_algo.append(algo_time)

        if times_baseline and times_algo:
            plt.figure(figsize=(10, 6))
            plt.scatter(times_baseline, times_algo, alpha=0.5)
            plt.xlabel("Baseline Execution Time (s)")
            plt.ylabel(f"{algo} Execution Time (s)")
            plt.title(f"Execution Time Comparison: Baseline vs {algo} (Excluding Timeouts)")
            max_time = max(max(times_baseline), max(times_algo))
            plt.plot([0, max_time], [0, max_time], "r--")
            plt.savefig(os.path.join(output_dir, f"execution_time_scatter_{algo}.png"))
            plt.close()
        else:
            print(f"No execution time data to plot for {algo}")

    print(f"Plots have been saved in {output_dir}")

In [ ]:
# Specify your directories here
baseline_dir = "runs/original_unmodified"
other_algo_dirs = {
    "avx256": "runs/avx256",
    "unroll256": "runs/unroll256",
    "avx128": "runs/avx128",
    # Add more algorithms as needed
}

output_dir = "analysis_output"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Analyze results
print("Analyzing baseline results...")
baseline_results = analyze_results(baseline_dir)
print(f"Found {len(baseline_results)} results in baseline.")

other_results = {}
for name, dir in other_algo_dirs.items():
    print(f"Analyzing results for {name}...")
    res = analyze_results(dir)
    print(f"Found {len(res)} results for {name}.")
    other_results[name] = res

# Compare results
comparison = compare_results(baseline_results, other_results)
print(f"Comparison data prepared with {len(comparison)} entries.")

# Write CSV
csv_file = os.path.join(output_dir, "comparison_results.csv")
write_comparison_csv(comparison, csv_file, list(other_algo_dirs.keys()))

# Plot results
plot_results(comparison, output_dir, list(other_algo_dirs.keys()))

print("Analysis complete!")